In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import random
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Flatten,Dense,Conv2D,MaxPooling2D
from tensorflow.keras.optimizers import SGD
from concurrent.futures import ThreadPoolExecutor

In [2]:
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
print("Environment set. TF restricted to 1 thread per instance to allow 12-thread concurrency.")

Environment set. TF restricted to 1 thread per instance to allow 12-thread concurrency.


In [3]:
# Splitting and normalising the data
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
# Adding channel dimension for CNN
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

In [4]:
# Creating IID Partition
def iid_partition(x, y, num_clients):
    idx = np.random.permutation(len(x))
    x_shuffled, y_shuffled = x[idx], y[idx]
    x_parts = np.array_split(x_shuffled, num_clients)
    y_parts = np.array_split(y_shuffled, num_clients)
    partitions = list(zip(x_parts, y_parts))
    return partitions

In [5]:
# Creating NON-IID Partition using 2 label skewness
def non_iid_partition(x, y, num_clients):
    sorted_idx = np.argsort(y)
    X_sorted, Y_sorted = x[sorted_idx], y[sorted_idx]
    num_shards = 2 * num_clients
    size_shards = len(x) // num_shards
    shards_idx = np.random.permutation(num_shards)
    partitions = []
    for i in range(num_clients):
        s1, s2 = shards_idx[2*i], shards_idx[2*i+1]
        X_client = np.concatenate((X_sorted[s1*size_shards:(s1+1)*size_shards], X_sorted[s2*size_shards:(s2+1)*size_shards]), axis=0)
        Y_client = np.concatenate((Y_sorted[s1*size_shards:(s1+1)*size_shards], Y_sorted[s2*size_shards:(s2+1)*size_shards]), axis=0)
        idx = np.random.permutation(len(X_client))
        partitions.append((X_client[idx], Y_client[idx]))
    return partitions

In [6]:
# Model
def cnn_model():
    model = Sequential([
        Input(shape=(28, 28, 1)),
        Conv2D(32, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax'),
    ])
    model.compile(optimizer=SGD(learning_rate=0.01), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [7]:
# local training for clients
def local_training(client_data,global_weights):
    X_local,Y_local=client_data
    num_sample=len(X_local)
    model=cnn_model()
    model.set_weights(global_weights)
    model.fit(X_local,Y_local,epochs=2,batch_size=32,verbose=0)
    local_weights = model.get_weights()
    return local_weights, num_sample

In [8]:
# Server
class federated_server:
    def __init__(self,X_test,Y_test):
        self.global_model=cnn_model()
        self.X_test=X_test
        self.Y_test=Y_test
    def get_global(self):
        return self.global_model.get_weights()
    def set_global(self,weights):
        self.global_model.set_weights(weights)
    def aggregate_weights(self,client_updates):
        tot_sample=sum(num_samples for _,num_samples in client_updates)
        global_weights=self.get_global()
        new_weights=[np.zeros_like(w) for w in global_weights]
        for local_weights,num_samples in client_updates:
            w_factor=num_samples/tot_sample
            for i in range(len(new_weights)):
                new_weights[i]+=local_weights[i]*w_factor
        self.set_global(new_weights)
    def evaluate_global(self):
        loss, accuracy = self.global_model.evaluate(self.X_test, self.Y_test, verbose=1)
        return loss, accuracy

In [9]:
def run_simulation(partitions, X_test,Y_test,num_clients,rounds,participation_rate):
    server=federated_server(X_test,Y_test)
    max_workers=min(num_clients,12)
    history=[]
    for r in range(1,rounds+1):
        global_weights=server.global_model.get_weights()
        num_active=max(1,int(num_clients*participation_rate))
        active_clients=random.sample(range(num_clients),num_active)
        client_updates=[]
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures=[executor.submit(local_training,partitions[i],global_weights) for i in active_clients]
            for future in futures:
                client_updates.append(future.result())
        server.aggregate_weights(client_updates)
        loss,acc=server.evaluate_global()
        history.append({
        'round': r,
        'loss': loss,
        'acc': acc
    })
        print(f"Round {r} | Accuracy: {acc} | Loss: {loss}")
    return history

In [11]:
CLIENTS=10
IID_ROUNDS=20
PARTICIPATION_RATE=1
print(f"===RUNNING IID FEDERATED LEARNING 100% PARTICIPATION({CLIENTS} Clients)===")
iid_parts=iid_partition(X_train,Y_train,CLIENTS)
iid_history1=run_simulation(iid_parts,X_test,Y_test,CLIENTS,IID_ROUNDS,PARTICIPATION_RATE)

===RUNNING IID FEDERATED LEARNING 100% PARTICIPATION(10 Clients)===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8740 - loss: 0.4511
Round 1 | Accuracy: 0.8740000128746033 | Loss: 0.45110997557640076
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9186 - loss: 0.2811
Round 2 | Accuracy: 0.9186000227928162 | Loss: 0.2810550034046173
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9341 - loss: 0.2240
Round 3 | Accuracy: 0.9340999722480774 | Loss: 0.22397065162658691
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9460 - loss: 0.1810
Round 4 | Accuracy: 0.9459999799728394 | Loss: 0.1809508353471756
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9534 - loss: 0.1531
Round 5 | Accuracy: 0.9534000158309937 | Loss: 0.15308374166488647
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9592 - loss: 0.1334
Round 6 | Accuracy: 0.9592000246047974 | Loss: 0.13336248695850372
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9662 - loss: 0.1157


In [13]:
NON_IID_ROUNDS=50
non_iid_parts=non_iid_partition(X_train,Y_train,CLIENTS)
print(f"===RUNNING NON-IID FEDERATED LEARNING 100% PARTICIPATION({CLIENTS} Clients)===")
non_iid_history1=run_simulation(non_iid_parts,X_test,Y_test,CLIENTS,NON_IID_ROUNDS,PARTICIPATION_RATE)

===RUNNING NON-IID FEDERATED LEARNING 100% PARTICIPATION(10 Clients)===
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.3077 - loss: 1.9833
Round 1 | Accuracy: 0.3077000081539154 | Loss: 1.9833078384399414
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5025 - loss: 1.3692
Round 2 | Accuracy: 0.5024999976158142 | Loss: 1.3692148923873901
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.6203 - loss: 1.0627
Round 3 | Accuracy: 0.6202999949455261 | Loss: 1.0626839399337769
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7067 - loss: 0.8460
Round 4 | Accuracy: 0.7067000269889832 | Loss: 0.8459792137145996
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7454 - loss: 0.7407
Round 5 | Accuracy: 0.7454000115394592 | Loss: 0.740652322769165
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7672 - loss: 0.6737
Round 6 | Accuracy: 0.7671999931335449 | Loss: 0.673662006855011
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7985 - loss: 0.5871
Round 7 |

In [14]:
IID_ROUNDS=50
PARTICIPATION_RATE=0.5
print(f"===RUNNING IID FEDERATED LEARNING 100% PARTICIPATION({CLIENTS} Clients)===")
iid_history2=run_simulation(iid_parts,X_test,Y_test,CLIENTS,IID_ROUNDS,PARTICIPATION_RATE)
NON_IID_ROUNDS=100
print(f"===RUNNING NON-IID FEDERATED LEARNING 100% PARTICIPATION({CLIENTS} Clients)===")
non_iid_history2=run_simulation(non_iid_parts,X_test,Y_test,CLIENTS,NON_IID_ROUNDS,PARTICIPATION_RATE)

===RUNNING IID FEDERATED LEARNING 100% PARTICIPATION(10 Clients)===
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8626 - loss: 0.4879
Round 1 | Accuracy: 0.8626000285148621 | Loss: 0.48790624737739563
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9147 - loss: 0.2840
Round 2 | Accuracy: 0.9146999716758728 | Loss: 0.2840297818183899
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9324 - loss: 0.2277
Round 3 | Accuracy: 0.9323999881744385 | Loss: 0.2277224063873291
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9490 - loss: 0.1788
Round 4 | Accuracy: 0.9490000009536743 | Loss: 0.17883630096912384
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9574 - loss: 0.1483
Round 5 | Accuracy: 0.9574000239372253 | Loss: 0.1482611745595932
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9590 - loss: 0.1345
Round 6 | Accuracy: 0.9589999914169312 | Loss: 0.13447481393814087
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9659 - loss: 0.1136
Round 7 

In [15]:
IID_ROUNDS=50
PARTICIPATION_RATE=0.3
print(f"===RUNNING IID FEDERATED LEARNING 100% PARTICIPATION({CLIENTS} Clients)===")
iid_history3=run_simulation(iid_parts,X_test,Y_test,CLIENTS,IID_ROUNDS,PARTICIPATION_RATE)
NON_IID_ROUNDS=100
print(f"===RUNNING NON-IID FEDERATED LEARNING 100% PARTICIPATION({CLIENTS} Clients)===")
non_iid_history3=run_simulation(non_iid_parts,X_test,Y_test,CLIENTS,NON_IID_ROUNDS,PARTICIPATION_RATE)

===RUNNING IID FEDERATED LEARNING 100% PARTICIPATION(10 Clients)===
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8632 - loss: 0.4936
Round 1 | Accuracy: 0.8632000088691711 | Loss: 0.4935545027256012
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9136 - loss: 0.2922
Round 2 | Accuracy: 0.9136000275611877 | Loss: 0.2922249138355255
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9341 - loss: 0.2291
Round 3 | Accuracy: 0.9340999722480774 | Loss: 0.22914674878120422
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9370 - loss: 0.2015
Round 4 | Accuracy: 0.9369999766349792 | Loss: 0.201542466878891
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9544 - loss: 0.1533
Round 5 | Accuracy: 0.9544000029563904 | Loss: 0.15329274535179138
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9596 - loss: 0.1332
Round 6 | Accuracy: 0.9595999717712402 | Loss: 0.13324452936649323
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9628 - loss: 0.1245
Ro

In [18]:
np.save("iid_history1.npy", iid_history1)
np.save("non_iid_history1.npy", non_iid_history1)

np.save("iid_history2.npy", iid_history2)
np.save("non_iid_history2.npy", non_iid_history2)

np.save("iid_history3.npy", iid_history3)
np.save("non_iid_history3.npy", non_iid_history3)